In [24]:
import requests
import numpy as np
import pandas as pd 
import pprint
import sqlite3
import json
from datetime import datetime

In [25]:
def create_database():
    conn = sqlite3.connect("macro_ml.db")
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS macro_data (
        date TEXT,
        indicator TEXT,
        value REAL,
        PRIMARY KEY (date, indicator)
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS price_data (
            date TEXT,
            ticker TEXT,
            adjusted_close REAL,
            volume INTEGER,
            PRIMARY KEY (date, ticker) 
        )
    """)

    conn.commit()
    conn.close()
    print("Database and tables initialized!")

def list_tables():
    conn = sqlite3.connect("macro_ml.db")
    cursor = conn.cursor()
    cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type='table'
        AND name NOT LIKE 'sqlite_%'; 
    """)

    tables = cursor.fetchall()
    conn.close()

    print(tables)
    print("---------------")
    print("Tables in macro_ml.db:")
    for table in tables:
        print(f"- {table[0]}")

In [4]:
create_database()
list_tables()

Database and tables initialized!
[('macro_data',), ('price_data',)]
---------------
Tables in macro_ml.db:
- macro_data
- price_data


In [26]:
API_KEY = "AYBALAEUTJH8V4VW"
URL_SPY = f'https://www.alphavantage.co/query?function=TIME_SERIES_MONTHLY&symbol=SPY&apikey={API_KEY}'
SPY_DATA = requests.get(URL_SPY).json()

In [27]:
SPY_time_series = SPY_DATA.get("Monthly Time Series")

In [28]:
SPY_input = []
for date, monthly_data in SPY_time_series.items():
    price = float(monthly_data['4. close'])
    volume = int(monthly_data['5. volume'])
    SPY_input.append((date, 'SPY', price, volume))

In [29]:
with sqlite3.connect("macro_ml.db") as conn:

    cursor = conn.cursor() 
    cursor.executemany("""
        INSERT OR REPLACE INTO price_data (date, ticker, adjusted_close, volume)
        VALUES(?,?,?,?)
    """, SPY_input)
conn.close()

with sqlite3.connect("macro_ml.db") as conn:
    SPY_df = pd.read_sql("SELECT * FROM price_data", conn)
conn.close()

In [30]:
URL_CPI = f'https://www.alphavantage.co/query?function=CPI&interval=monthly&apikey={API_KEY}'
CPI_DATA = requests.get(URL_CPI)

In [31]:
CPI_data_monthly = CPI_DATA.json().get("data")

CPI_comp = []

for monthly in CPI_data_monthly:
    date_str = monthly['date']
    raw_value = monthly['value']
    CPI = float(raw_value) if raw_value != "." else None 
    converted_date = datetime.strptime(date_str, "%Y-%m-%d")
    if converted_date > datetime(2000, 1, 1):
        CPI_comp.append((date_str, "CPI", CPI))

In [117]:
with sqlite3.connect("macro_ml.db") as conn:

    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO macro_data (date, indicator, value)
        VALUES(?, ?, ?)
    """, CPI_comp)
conn.close()

with sqlite3.connect("macro_ml.db") as conn:
    CPI_df = pd.read_sql("SELECT * FROM macro_data WHERE indicator='CPI'", conn)
conn.close()
print(CPI_df)

           date indicator    value
0    2026-08-01       CPI  334.980
1    2026-07-01       CPI  333.918
2    2026-06-01       CPI  333.952
3    2026-05-01       CPI  335.123
4    2026-04-01       CPI  333.020
..          ...       ...      ...
314  2000-06-01       CPI  172.400
315  2000-05-01       CPI  171.500
316  2000-04-01       CPI  171.300
317  2000-03-01       CPI  171.200
318  2000-02-01       CPI  169.800

[319 rows x 3 columns]


In [33]:
URL_UNEMPLOYMENT = 'https://www.alphavantage.co/query?function=UNEMPLOYMENT&apikey={API_KEY}'
UNEMPLOYMENT = requests.get(URL_UNEMPLOYMENT)

In [34]:
UNEMPLOYMENT_monthly = UNEMPLOYMENT.json().get("data")

UNEMP_comp = []

for unemp in UNEMPLOYMENT_monthly:
    date_str = unemp['date']
    raw_val = unemp['value']
    monthly_unemp = float(raw_val) if raw_val != "." else None
    converted_date = datetime.strptime(date_str, "%Y-%m-%d")
    if converted_date > datetime(2000,1,1):
        UNEMP_comp.append((date_str, "UNEMP", monthly_unemp))

In [139]:
with sqlite3.connect("macro_ml.db") as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO macro_data (date, indicator, value)
        VALUES(?, ?, ?)
    """, UNEMP_comp)

conn.close()

with sqlite3.connect("macro_ml.db") as conn:
    UNEMP_df = pd.read_sql("SELECT * FROM macro_data WHERE indicator='UNEMP'", conn)
conn.close()

print(UNEMP_df)

           date indicator  value
0    2026-08-01     UNEMP    4.1
1    2026-07-01     UNEMP    4.1
2    2026-06-01     UNEMP    4.2
3    2026-05-01     UNEMP    4.3
4    2026-04-01     UNEMP    4.3
..          ...       ...    ...
314  2000-06-01     UNEMP    4.0
315  2000-05-01     UNEMP    4.0
316  2000-04-01     UNEMP    3.8
317  2000-03-01     UNEMP    4.0
318  2000-02-01     UNEMP    4.1

[319 rows x 3 columns]


In [36]:
URL_FRR = 'https://www.alphavantage.co/query?function=FEDERAL_FUNDS_RATE&interval=monthly&apikey={API_KEY}'
FRR = requests.get(URL_FRR).json()

In [132]:
FRR_monthly_data = FRR.get("data")

FRR_compiled = []

for month in FRR_monthly_data:
    date_str = month['date']
    raw_val = float(month['value'])
    rate = raw_val if raw_val != "." else None 
    
    if date_str >= "2000-01-01":
        FRR_compiled.append((date_str, "Fed_Rate", rate))    
        # print(date_str, rate)


pprint.pprint(FRR_compiled)
# print(FRR_pd)
# print(FRR_pd['value'].unique())
# pprint.pprint(FRR_monthly_data)

[('2026-08-01', 'Fed_Rate', 3.63),
 ('2026-07-01', 'Fed_Rate', 3.63),
 ('2026-06-01', 'Fed_Rate', 3.63),
 ('2026-05-01', 'Fed_Rate', 3.63),
 ('2026-04-01', 'Fed_Rate', 3.64),
 ('2026-03-01', 'Fed_Rate', 3.64),
 ('2026-02-01', 'Fed_Rate', 3.64),
 ('2026-01-01', 'Fed_Rate', 3.64),
 ('2025-12-01', 'Fed_Rate', 3.72),
 ('2025-11-01', 'Fed_Rate', 3.88),
 ('2025-10-01', 'Fed_Rate', 4.09),
 ('2025-09-01', 'Fed_Rate', 4.22),
 ('2025-08-01', 'Fed_Rate', 4.33),
 ('2025-07-01', 'Fed_Rate', 4.33),
 ('2025-06-01', 'Fed_Rate', 4.33),
 ('2025-05-01', 'Fed_Rate', 4.33),
 ('2025-04-01', 'Fed_Rate', 4.33),
 ('2025-03-01', 'Fed_Rate', 4.33),
 ('2025-02-01', 'Fed_Rate', 4.33),
 ('2025-01-01', 'Fed_Rate', 4.33),
 ('2024-12-01', 'Fed_Rate', 4.48),
 ('2024-11-01', 'Fed_Rate', 4.64),
 ('2024-10-01', 'Fed_Rate', 4.83),
 ('2024-09-01', 'Fed_Rate', 5.13),
 ('2024-08-01', 'Fed_Rate', 5.33),
 ('2024-07-01', 'Fed_Rate', 5.33),
 ('2024-06-01', 'Fed_Rate', 5.33),
 ('2024-05-01', 'Fed_Rate', 5.33),
 ('2024-04-01', 'Fed

In [138]:
with sqlite3.connect("macro_ml.db") as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO macro_data (date, indicator, value)
        VALUES(?, ?, ?)
    """, FRR_compiled)
conn.close() 

with sqlite3.connect("macro_ml.db") as conn:
    FRR_df = pd.read_sql("SELECT * FROM macro_data WHERE indicator='Fed_Rate'", conn)

print(FRR_df)

           date indicator  value
0    2026-08-01  Fed_Rate   3.63
1    2026-07-01  Fed_Rate   3.63
2    2026-06-01  Fed_Rate   3.63
3    2026-05-01  Fed_Rate   3.63
4    2026-04-01  Fed_Rate   3.64
..          ...       ...    ...
315  2000-05-01  Fed_Rate   6.27
316  2000-04-01  Fed_Rate   6.02
317  2000-03-01  Fed_Rate   5.85
318  2000-02-01  Fed_Rate   5.73
319  2000-01-01  Fed_Rate   5.45

[320 rows x 3 columns]


In [69]:
URL_10yr = url = 'https://www.alphavantage.co/query?function=TREASURY_YIELD&interval=monthly&maturity=10year&apikey={API_key}'
T10yr = requests.get(URL_10yr).json()

In [74]:
T10yr_monthly = T10yr.get("data")
T10yr_compiled = []

for month in T10yr_monthly:
    date_str = month['date']
    raw_val = month['value']
    rate = raw_val if raw_val != "." else None
    if date_str >= "2000-01-01":
        T10yr_compiled.append((date_str, "10yr", rate))        

In [137]:
with sqlite3.connect("macro_ml.db") as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO macro_data (date, indicator, value)
        VALUES(?, ?, ?) 
    """, T10yr_compiled)
conn.close()

with sqlite3.connect("macro_ml.db") as conn:
    trea10_df = pd.read_sql("SELECT * FROM macro_data WHERE indicator='10yr'", conn)

print(trea10_df)

           date indicator  value
0    2026-08-01      10yr   4.68
1    2026-07-01      10yr   4.60
2    2026-06-01      10yr   4.47
3    2026-05-01      10yr   4.48
4    2026-04-01      10yr   4.32
..          ...       ...    ...
315  2000-05-01      10yr   6.44
316  2000-04-01      10yr   5.99
317  2000-03-01      10yr   6.26
318  2000-02-01      10yr   6.52
319  2000-01-01      10yr   6.66

[320 rows x 3 columns]


In [79]:
URL_RETAIL = 'https://www.alphavantage.co/query?function=RETAIL_SALES&apikey={API_Key}'
RETAIL_SALES = requests.get(URL_RETAIL).json()

In [86]:
monthly_retail = RETAIL_SALES.get("data")
retail_compiled = []

for month in monthly_retail:
    date_str = month['date']
    rate = month['value']
    if date_str >= "2000-01-01":
        retail_compiled.append((date_str, "retail", rate))

In [140]:
with sqlite3.connect("macro_ml.db") as conn:
    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO macro_data (date, indicator, value)
        VALUES(?, ?, ?) 
    """, retail_compiled)
conn.close()

with sqlite3.connect("macro_ml.db") as conn:
    retail_df = pd.read_sql("SELECT * FROM macro_data WHERE indicator='retail'", conn)

print(retail_df)

           date indicator     value
0    2026-08-01    retail  676204.0
1    2026-07-01    retail  677249.0
2    2026-06-01    retail  671109.0
3    2026-05-01    retail  685550.0
4    2026-04-01    retail  654070.0
..          ...       ...       ...
315  2000-05-01    retail  251700.0
316  2000-04-01    retail  233673.0
317  2000-03-01    retail  247840.0
318  2000-02-01    retail  221662.0
319  2000-01-01    retail  208890.0

[320 rows x 3 columns]


In [158]:
### the macro_data table in SQL

with sqlite3.connect("macro_ml.db") as conn:
    total_macro_df = pd.read_sql("SELECT* FROM macro_data", conn)
conn.close()

macro_wide_df = total_macro_df.pivot(index="date", columns="indicator", values="value").reset_index()
macro_wide_df['year_month'] = pd.to_datetime(macro_wide_df['date']).dt.to_period("M")
# print(macro_wide_df)

with sqlite3.connect("macro_ml.db") as conn:
    SPY_df = pd.read_sql("SELECT * FROM price_data", conn)

SPY_df['year_month'] = pd.to_datetime(SPY_df['date']).dt.to_period("M")
SPY_df = SPY_df.iloc[1:]
# print(SPY_df)

combined_df = pd.merge(macro_wide_df, SPY_df, how="left", on="year_month")
useable_df = combined_df.copy()
useable_df = useable_df.drop(columns=['date_y', 'date_x'])
# prefered_order = ['year_month', 'ticker', 'adjusted_close', 'volume', '10yr', 'CPI', 'Fed_Rate', 'UNEMPY', 'retail']
# useable_df = useable_df[prefered_order]
print(useable_df)

     10yr      CPI  Fed_Rate  UNEMP    retail year_month ticker  \
0    6.66      NaN      5.45    NaN  208890.0    2000-01    SPY   
1    6.52  169.800      5.73    4.1  221662.0    2000-02    SPY   
2    6.26  171.200      5.85    4.0  247840.0    2000-03    SPY   
3    5.99  171.300      6.02    3.8  233673.0    2000-04    SPY   
4    6.44  171.500      6.27    4.0  251700.0    2000-05    SPY   
..    ...      ...       ...    ...       ...        ...    ...   
315  4.32  333.020      3.64    4.3  654070.0    2026-04    SPY   
316  4.48  335.123      3.63    4.3  685550.0    2026-05    SPY   
317  4.47  333.952      3.63    4.2  671109.0    2026-06    SPY   
318  4.60  333.918      3.63    4.1  677249.0    2026-07    SPY   
319  4.68  334.980      3.63    4.1  676204.0    2026-08    SPY   

     adjusted_close      volume  
0          139.5625   156770800  
1          137.4375   186938300  
2          150.3750   247594900  
3          145.0937   229246200  
4          142.8125   161